<a href="https://colab.research.google.com/github/thithikhine1506/OpenPI-VLA-with-Kinova-Robotic-Arm/blob/main/OpenPI_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!python --version

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone --recurse-submodules -b kinova-gen3 https://github.com/thithikhine1506/openpi.git /content/openpi
%cd /content/openpi
!GIT_LFS_SKIP_SMUDGE=1 ~/.local/bin/uv sync
!GIT_LFS_SKIP_SMUDGE=1 ~/.local/bin/uv pip install -e .

In [ ]:
%cd /content/openpi
!uv python install 3.11
!GIT_LFS_SKIP_SMUDGE=1 uv sync --python 3.11
!GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .

In [ ]:
!uv run python -c "import openpi, jax; print('openpi ok, jax', jax.__version__); print(jax.devices())"

In [ ]:
!uv run huggingface-cli login

**Cell 4 — norm stats**

In [ ]:
!uv run scripts/compute_norm_stats.py --config-name pi05_kinova_gen3_lora

**Cell 5 — short test run**

In [ ]:
!uv run scripts/train.py pi05_kinova_gen3_lora \
    --exp-name=kinova_test --overwrite --num-train-steps 200

**Cell 6 — checkpoints to Drive, then the real run**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/openpi_checkpoints
!rm -rf /content/openpi/checkpoints
!ln -sfn /content/drive/MyDrive/openpi_checkpoints /content/openpi/checkpoints

In [ ]:
!uv run scripts/train.py pi05_kinova_gen3_lora \
    --exp-name=kinova_pick_place --overwrite

**If the session drops, re-run the install cells, HF login, and norm stats, then:**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/

In [ ]:
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
!ls -la /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/
!df -h /content/drive

In [ ]:
!du -sh /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/*/
!ls -la /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/12000/
!ls -la /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/12000/params/ 2>/dev/null | head

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone --recurse-submodules -b kinova-gen3 https://github.com/thithikhine1506/openpi.git /content/openpi
%cd /content/openpi
!uv python install 3.11
!GIT_LFS_SKIP_SMUDGE=1 uv sync --python 3.11
!uv run huggingface-cli login
!uv run scripts/compute_norm_stats.py --config-name pi05_kinova_gen3_lora

In [ ]:
!uv run scripts/serve_policy.py policy:checkpoint \
    --policy.config pi05_kinova_gen3_lora \
    --policy.dir /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/11000

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/

In [ ]:
!mkdir -p /content/ckpt
!cp -r /content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/11000 /content/ckpt/
!du -sh /content/ckpt/11000

In [ ]:
!cd /content/ckpt && tar czf /content/ckpt_11000.tar.gz 11000
!ls -lh /content/ckpt_11000.tar.gz

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone --recurse-submodules -b kinova-gen3 https://github.com/thithikhine1506/openpi.git /content/openpi
%cd /content/openpi
!uv python install 3.11
!GIT_LFS_SKIP_SMUDGE=1 uv sync --python 3.11

In [ ]:
!uv run huggingface-cli login
!uv run scripts/compute_norm_stats.py --config-name pi05_kinova_gen3_lora

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyngrok

In [ ]:
server.terminate()

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import subprocess, time, re

server = subprocess.Popen([
    "uv", "run", "scripts/serve_policy.py", "policy:checkpoint",
    "--policy.config", "pi05_kinova_gen3_lora",
    "--policy.dir", "/content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/11000",
], cwd="/content/openpi")

time.sleep(90)   # model load

tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "tcp://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in tun.stdout:
    print(line, end="")
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> TUNNEL URL:", m.group(0))
        break

In [ ]:
cd ~/kinova_collect
source ~/kinova_env/bin/activate
pip install -e ~/openpi/packages/openpi-client
python3 -c "from openpi_client import websocket_client_policy; print('client ok')"

In [ ]:
print("server running:", server.poll() is None)

In [ ]:
tun.terminate()

import subprocess, re
tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in tun.stdout:
    print(line, end="")
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> TUNNEL URL:", m.group(0))
        break

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone -b kinova-gen3 https://github.com/thithikhine1506/openpi.git /content/openpi
%cd /content/openpi
!uv python install 3.11
!GIT_LFS_SKIP_SMUDGE=1 uv sync --python 3.11
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, time, re

server = subprocess.Popen([
    "uv", "run", "scripts/serve_policy.py", "policy:checkpoint",
    "--policy.config", "pi05_kinova_gen3_lora",
    "--policy.dir", "/content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/11000",
], cwd="/content/openpi")

time.sleep(120)

tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in tun.stdout:
    print(line, end="")
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> TUNNEL URL:", m.group(0))
        break

In [ ]:
print("server:", server.poll() is None)
print("tunnel:", tun.poll() is None)

In [ ]:
tun.terminate()
import subprocess, re, time
time.sleep(2)
tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in tun.stdout:
    print(line, end="")
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> TUNNEL URL:", m.group(0))
        break

In [ ]:
!curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8000 || echo "nothing listening"

In [ ]:
server.terminate(); tun.terminate()
import subprocess, time, re
time.sleep(5)

server = subprocess.Popen([
    "uv", "run", "scripts/serve_policy.py", "policy:checkpoint",
    "--policy.config", "pi05_kinova_gen3_lora",
    "--policy.dir", "/content/drive/MyDrive/openpi_checkpoints/pi05_kinova_gen3_lora/kinova_pick_place/5000",
], cwd="/content/openpi", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in server.stdout:
    print(line, end="")
    if "server listening" in line:
        print(">>> SERVER READY")
        break

tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in tun.stdout:
    print(line, end="")
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> TUNNEL URL:", m.group(0))
        break